# ConstructIA V3.1 — Avance de proyecto 2
## Curación de imágenes e integración texto–imagen mejorada

Esta versión **conserva el experimento original V3** y añade tres cambios: (a) excluye de manera preliminar las fotografías claramente ajenas o poco representativas identificadas en las capturas del primer entrenamiento, (b) genera una galería del dataset para permitir una auditoría completa y (c) incorpora reglas comerciales explicables **sin modificar la predicción SVM**.

**Advertencia metodológica:** como ya revisamos los resultados de prueba de V3, las métricas que eventualmente se obtengan con V3.1 serán parte de una iteración exploratoria. No las presentaremos como una nueva evaluación externa completamente independiente. Antes de uso comercial necesitaremos un conjunto de fotografías nuevo, que no haya intervenido en este desarrollo.


## 1. Preparar entorno
Elige **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** si está disponible. **Ejecuta las celdas por orden y detente en la sección 3A**, donde revisarás las imágenes antes de confirmar la auditoría. No elijas «Ejecutar todas» sin revisar primero: hemos agregado una comprobación explícita.

El cuaderno requiere conexión a Internet para descargar fotografías con su atribución y los pesos ImageNet de ResNet50. Usa únicamente imágenes autorizadas; nunca publiques datos reales de clientes.


In [ ]:
%pip -q install requests pillow pandas scikit-learn matplotlib torch torchvision tqdm imagehash nltk
import os, sys, json, re, html, hashlib, random, time, pathlib
import numpy as np, pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from PIL import Image, ImageOps
import imagehash
import torch, torchvision
from torchvision import transforms, models
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = pathlib.Path('/content/ConstructIA_V3')
IMG_DIR = ROOT / 'imagenes'
ROOT.mkdir(exist_ok=True, parents=True)
IMG_DIR.mkdir(exist_ok=True, parents=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__, '| dispositivo:', DEVICE)

## 2. Reunir fotografías con fuente y licencia
Usaremos categorías de **Wikimedia Commons**. La meta inicial son **55 imágenes válidas por clase** (ajustable más adelante). Se guardan nombre, autor, licencia, URL de licencia y página de origen. Si hay fallos de descarga, el programa intenta con otras fotos. No atribuyas a clientes reales las fotos de terceros.

Categorías de referencia: [Backhoe loaders](https://commons.wikimedia.org/wiki/Category:Backhoe_loaders), [Scissor lifts](https://commons.wikimedia.org/wiki/Category:Scissor_lifts), [Skid-steer loaders](https://commons.wikimedia.org/wiki/Category:Skid-steer_loaders).

In [ ]:
API = 'https://commons.wikimedia.org/w/api.php'
FUENTES = {
  'retroexcavadora': ['Backhoe loaders', 'JCB backhoe loaders', 'Caterpillar backhoe loaders'],
  'plataforma_elevadora': ['Scissor lifts', 'Genie Industries scissor lifts', 'JLG scissor lifts'],
  'minicargador': ['Skid-steer loaders', 'Case skid-steer loaders', 'Caterpillar skid-steer loaders'],
}
OBJETIVO_POR_CLASE = 55
MINIMO_POR_CLASE = 24

sesion = requests.Session()
sesion.headers.update({'User-Agent': 'ConstructIA-academic-vision/0.1 (https://github.com/chrisavm24-dot/ConstructIA; classroom research)', 'Accept': 'application/json'})
retry = Retry(total=3, backoff_factor=1.5, status_forcelist=[429,500,502,503,504])
sesion.mount('https://', HTTPAdapter(max_retries=retry))

def api_imagenes(categoria):
    parametros = {
        'action':'query', 'generator':'categorymembers',
        'gcmtitle':'Category:'+categoria, 'gcmtype':'file', 'gcmlimit': '200',
        'prop':'imageinfo', 'iiprop':'url|mime|size|extmetadata',
        'iiurlwidth':'640', 'format':'json', 'formatversion':'2'
    }
    paginas=[]
    for _ in range(8):
        respuesta=sesion.get(API, params=parametros, timeout=45)
        respuesta.raise_for_status()
        objeto=respuesta.json()
        if 'error' in objeto: raise RuntimeError(f'Commons API: {objeto["error"]}')
        paginas.extend(objeto.get('query',{}).get('pages',[]))
        siguiente=objeto.get('continue')
        if not siguiente: break
        parametros.update(siguiente)
    return paginas

def meta_valor(meta, clave):
    return html.unescape(re.sub('<[^>]+>', '', meta.get(clave,{}).get('value',''))).strip()

def licencia_permitida(nombre):
    n=nombre.lower().replace('-', ' ').strip()
    return (n.startswith('cc by ') or n.startswith('cc by sa ') or n.startswith('cc0')
            or n.startswith('public domain') or n.startswith('pd '))

def candidatos_por_clase(clase):
    candidatos={}
    for categoria in FUENTES[clase]:
        try: paginas=api_imagenes(categoria)
        except requests.RequestException as exc:
            print('Problema al consultar', categoria, ':',str(exc)[:140]); continue
        for pagina in paginas:
            titulo=pagina.get('title','')
            if not titulo.startswith('File:') or titulo in candidatos: continue
            info=(pagina.get('imageinfo') or [{}])[0]
            if info.get('mime','') not in ('image/jpeg','image/png','image/webp'): continue
            meta=info.get('extmetadata') or {}
            licencia=meta_valor(meta,'LicenseShortName')
            if not licencia_permitida(licencia): continue
            url=info.get('thumburl') or info.get('url')
            if not url: continue
            candidatos[titulo]={'clase':clase,'titulo':titulo,'categoria_fuente':categoria,
                'imagen_url':url, 'pagina_fuente':info.get('descriptionurl',''),
                'autor':meta_valor(meta,'Artist')[:300], 'licencia':licencia,
                'licencia_url':meta_valor(meta,'LicenseUrl')}
        if len(candidatos)>=OBJETIVO_POR_CLASE*2: break
    items=list(candidatos.values())
    random.Random(SEED).shuffle(items)
    return items

lista_candidatos={clase:candidatos_por_clase(clase) for clase in FUENTES}
print('Fotografías candidatas con licencia:', {k:len(v) for k,v in lista_candidatos.items()})

In [ ]:
registros=[]
hashes=[]
titulos_usados=set()
for clase, candidatas in lista_candidatos.items():
    carpeta=IMG_DIR/clase; carpeta.mkdir(exist_ok=True,parents=True)
    exitos=0
    for item in tqdm(candidatas, desc='Descarga '+clase):
        if exitos >= OBJETIVO_POR_CLASE: break
        if item['titulo'] in titulos_usados: continue
        try:
            rr=sesion.get(item['imagen_url'], timeout=40)
            rr.raise_for_status()
            if len(rr.content)>8_000_000: continue
            from io import BytesIO
            foto=ImageOps.exif_transpose(Image.open(BytesIO(rr.content))).convert('RGB')
            if min(foto.size)<160: continue
            h=imagehash.phash(foto)
            # Eliminar fotografías idénticas o casi idénticas antes de separar los conjuntos
            if any(h-hprev<=3 for hprev in hashes): continue
            foto.thumbnail((768,768))
            nombre=hashlib.sha256(item['titulo'].encode('utf-8')).hexdigest()[:16]+'.jpg'
            ruta=carpeta/nombre
            foto.save(ruta,quality=85,optimize=True)
            registros.append({**item,'ruta':str(ruta),'hash_visual':str(h)})
            hashes.append(h); titulos_usados.add(item['titulo']); exitos+=1
        except (OSError, ValueError, requests.RequestException):
            continue
    print(clase,':',exitos,'imágenes descargadas')

manifest=pd.DataFrame(registros)
if manifest.empty: raise RuntimeError('No se descargaron imágenes. Revisa la conexión a Wikimedia Commons y las licencias.')
manifest.to_csv(ROOT/'origen_licencias_imagenes.csv',index=False,encoding='utf-8-sig')
print('TOTAL:',len(manifest),'| por categoría:',manifest['clase'].value_counts().to_dict())
if (manifest['clase'].value_counts() < MINIMO_POR_CLASE).any():
    raise RuntimeError('Una categoría quedó con pocas fotografías (<24). Revisa las categorías y permisos de descarga antes de entrenar.')

## 3. Revisión preliminar y descarte conservador
En las capturas de V3 identificamos ejemplos poco representativos: una intersección lejana, un primer plano de controles y un camión de transporte. Los títulos que coincidan con estas observaciones se excluyen **solo del manifiesto de entrenamiento**, pero sus imágenes originales siguen en disco para auditoría.

**No equivale a validar las 165 fotografías.** En la siguiente sección se generan galerías de **todas** las candidatas descargadas, con su ID, categoría y título abreviado. Si ves una foto errónea adicional, anota su ID en la lista `EXCLUIR_IDS_ADICIONALES` y vuelve a ejecutar esta celda y la siguiente.


In [ ]:
import unicodedata
if 'manifest_completo' not in globals():
    manifest_completo = manifest.copy(deep=True).reset_index(drop=True)
    manifest_completo['id_revision'] = np.arange(len(manifest_completo))

# Tres candidatas poco representativas detectadas en las capturas de V3.
# La coincidencia usa fragmentos del título de Commons, no predicciones de la CNN.
EXCLUIR_TITULOS_SUGERIDOS = [
    ('retroexcavadora', 'Zlín, Malenovice, třída 3.'),
    ('plataforma_elevadora', 'Scherenarbeitsbühne Bedienp'),
    ('minicargador', 'Harjanne Oy black Scania'),
]
EXCLUIR_IDS_ADICIONALES = []  # Ejemplo tras revisar galerías: [12, 75]
EXCLUIR_TITULOS_ADICIONALES = []  # Puedes escribir aquí títulos completos de Commons.

def clave_de_revision(t):
    return ''.join(c for c in unicodedata.normalize('NFD', str(t).casefold()) if unicodedata.category(c) != 'Mn')

ids_descartados=set(EXCLUIR_IDS_ADICIONALES)
coincidencias=[]
for clase, fragmento in EXCLUIR_TITULOS_SUGERIDOS:
    for _, fila in manifest_completo[manifest_completo['clase']==clase].iterrows():
        if clave_de_revision(fragmento) in clave_de_revision(fila['titulo']):
            ids_descartados.add(int(fila['id_revision']))
            coincidencias.append((int(fila['id_revision']), clase, fila['titulo']))
for titulo in EXCLUIR_TITULOS_ADICIONALES:
    ids_descartados.update(
        int(x) for x in manifest_completo.loc[manifest_completo['titulo']==titulo,'id_revision']
    )
manifest=manifest_completo[~manifest_completo['id_revision'].isin(ids_descartados)].copy().reset_index(drop=True)
print('Fotos descargadas inicialmente:', len(manifest_completo))
print('Coincidencias preliminares vistas en capturas:', len(coincidencias))
for id_, clase, titulo in coincidencias: print(' - Descartada',id_,clase,titulo)
print('Total excluidas:',len(manifest_completo)-len(manifest))
print('Por categoría que continúa:',manifest['clase'].value_counts().to_dict())
if manifest['clase'].nunique()!=3 or manifest['clase'].value_counts().min()<24:
    raise RuntimeError('Faltan fotografías suficientes en una categoría tras la curación.')
manifest.to_csv(ROOT/'origen_licencias_imagenes_curado.csv',index=False,encoding='utf-8-sig')
manifest_completo[manifest_completo['id_revision'].isin(ids_descartados)].to_csv(
    ROOT/'exclusiones_curacion_v3_1.csv',index=False,encoding='utf-8-sig'
)


## 3A. Auditoría visual del dataset
Esta celda genera páginas de 20 imágenes de **cada categoría** y muestra su **ID de revisión**. Revisa que la fotografía muestre claramente la máquina indicada; evita controles, carreteras sin máquina, camiones que transportan equipos u objetos muy pequeños.

Si detectas etiquetas incorrectas, toma nota de los IDs, vuelve a la sección 3 y agrégalos a `EXCLUIR_IDS_ADICIONALES`. Ejecuta nuevamente ambas celdas. **Al finalizar la revisión, cambia `AUDITORIA_VISUAL_CONFIRMADA` a `True`** en la celda siguiente. Este paso requiere juicio humano; el código no puede garantizar etiquetas correctas por sí solo.


In [ ]:
PAGINA_TAMANO=20
CARPETA_GALERIAS=ROOT/'galerias_revision'
CARPETA_GALERIAS.mkdir(exist_ok=True,parents=True)
for clase in FUENTES:
    filas=manifest[manifest['clase']==clase].sort_values('id_revision')
    for inicio in range(0,len(filas),PAGINA_TAMANO):
        muestra=filas.iloc[inicio:inicio+PAGINA_TAMANO]
        fig,axs=plt.subplots(4,5,figsize=(16,13))
        for ax in axs.flat: ax.axis('off')
        for ax,(_,fila) in zip(axs.flat,muestra.iterrows()):
            ax.imshow(Image.open(fila['ruta']).convert('RGB'))
            ax.set_title(f"#{int(fila['id_revision'])} {fila['titulo'][5:38]}",fontsize=7)
        fig.suptitle(f"{clase} — fotografías {inicio+1} a {inicio+len(muestra)}",fontsize=15)
        fig.tight_layout(rect=(0,0,1,0.98))
        ruta=CARPETA_GALERIAS/f"{clase}_{inicio//PAGINA_TAMANO+1:02d}.png"
        fig.savefig(ruta,dpi=105)
        plt.show()
        plt.close(fig)
print('Galerías guardadas en:',CARPETA_GALERIAS)
print('IMPORTANTE: fotografía pública no implica etiqueta correcta.')


## 3B. Confirmación explícita de auditoría
**No actives el siguiente indicador por comodidad.** Hazlo después de revisar las galerías y excluir las fotos claramente ajenas a las categorías. La curación es parcial si no se inspeccionan todas las imágenes.


In [ ]:
AUDITORIA_VISUAL_CONFIRMADA = False  # Cambia a True ÚNICAMENTE después de revisar las galerías.
if AUDITORIA_VISUAL_CONFIRMADA:
    print('Auditoría visual confirmada por el estudiante. Fotos que continúan:',len(manifest))
else:
    print('Auditoría pendiente. Revisa las galerías antes de entrenar.')


## 4. Separar entrenamiento, validación y prueba
Dividimos de manera estratificada **70% / 15% / 15%** (aproximadamente). Los archivos se descargaron y deduplicaron antes de dividirse. La prueba final queda reservada hasta terminar el entrenamiento.

Para resultados académicos honestos, evita cambiar imágenes o parámetros tras consultar la prueba final; si lo haces, documenta una nueva partición independiente.

In [ ]:
if not AUDITORIA_VISUAL_CONFIRMADA:
    raise RuntimeError('Aún falta revisar las galerías de la sección 3A. Confirma la auditoría en 3B antes de dividir los datos.')
train_df, temp_df = train_test_split(manifest,test_size=0.30,stratify=manifest['clase'],random_state=SEED)
val_df, test_df = train_test_split(temp_df,test_size=0.50,stratify=temp_df['clase'],random_state=SEED)
for nombre,d in [('entrenamiento',train_df),('validación',val_df),('prueba',test_df)]:
    print(nombre,':',len(d),d['clase'].value_counts().to_dict())
    d.to_csv(ROOT/f'{nombre}_manifest.csv',index=False,encoding='utf-8-sig')

## 5. ResNet50 con transferencia de aprendizaje
Usamos pesos preentrenados de ImageNet; congelamos la parte convolucional y entrenamos una nueva capa final de tres clases. Para mantener el experimento razonablemente reproducible fijamos semilla y registramos métricas de entrenamiento/validación. La red necesita fotos reales de cada clase; no genera imágenes artificiales para simular evaluación.

In [ ]:
CLASES=list(FUENTES.keys()); idx={et:i for i,et in enumerate(CLASES)}
transform_train=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.78,1.0)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
transform_eval=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class DatosFotos(Dataset):
    def __init__(self,df,transform): self.df=df.reset_index(drop=True);self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        m=self.df.iloc[i]; foto=Image.open(m['ruta']).convert('RGB')
        return self.transform(foto), idx[m['clase']]

train_loader=DataLoader(DatosFotos(train_df,transform_train),batch_size=16,shuffle=True,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
val_loader=DataLoader(DatosFotos(val_df,transform_eval),batch_size=16,shuffle=False,num_workers=2)
test_loader=DataLoader(DatosFotos(test_df,transform_eval),batch_size=16,shuffle=False,num_workers=2)
red=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in red.parameters(): param.requires_grad=False
red.fc=nn.Linear(red.fc.in_features,len(CLASES))
red=red.to(DEVICE)
criterio=nn.CrossEntropyLoss(); optimizador=torch.optim.Adam(red.fc.parameters(),lr=0.001)
print('ResNet50 con ImageNet | pesos convolucionales congelados | salida de',len(CLASES),'clases')

In [ ]:
EPOCAS=5
historial=[]; mejor_f1=-1; mejor_estado=None
def evaluar_loader(loader):
    red.eval(); y_real=[];y_pred=[];perdidas=[]
    with torch.no_grad():
        for xx,yy in loader:
            xx,yy=xx.to(DEVICE), yy.to(DEVICE)
            logits=red(xx); perdidas.append(criterio(logits,yy).item())
            y_real.extend(yy.cpu().tolist()); y_pred.extend(logits.argmax(1).cpu().tolist())
    return np.mean(perdidas), np.array(y_real), np.array(y_pred)

for epoca in range(1,EPOCAS+1):
    red.eval()  # conserva las estadísticas BatchNorm del extractor congelado
    red.fc.train()
    perdidas_train=[]
    for xx,yy in train_loader:
        xx,yy=xx.to(DEVICE),yy.to(DEVICE)
        optimizador.zero_grad()
        logits=red(xx)
        perdida=criterio(logits,yy)
        perdida.backward(); optimizador.step()
        perdidas_train.append(perdida.item())
    val_loss,yv,pv=evaluar_loader(val_loader)
    _,_,f1v,_=precision_recall_fscore_support(yv,pv,average='macro',zero_division=0)
    historial.append({'epoca':epoca,'loss_entrenamiento':np.mean(perdidas_train),'loss_validacion':val_loss,
                     'f1_macro_validacion':f1v})
    if f1v>mejor_f1:
        mejor_f1=f1v; mejor_estado={k:v.detach().cpu().clone() for k,v in red.state_dict().items()}
    print(f'Época {epoca}: loss train={np.mean(perdidas_train):.3f} | loss val={val_loss:.3f} | F1 macro val={f1v:.3f}')
if mejor_estado is None: raise RuntimeError('No se obtuvo un modelo entrenado')
red.load_state_dict(mejor_estado)
torch.save({'state_dict':mejor_estado,'clases':CLASES,'imagen_fuente':'Wikimedia Commons'},ROOT/'constructia_resnet50.pt')
print('Mejor F1 de validación:',round(mejor_f1,3),'| Modelo guardado en',ROOT/'constructia_resnet50.pt')
display(pd.DataFrame(historial))

## 6. Evaluación final del modelo visual
Esta es la evaluación de **prueba** reservada. Reportamos las cinco evidencias exigidas: accuracy, precision, recall, F1-score y matriz de confusión. Las métricas son de este dataset público pequeño, **no representan precisión comercial**.

In [ ]:
test_loss, yt, yp = evaluar_loader(test_loader)
pr,rc,f1,_=precision_recall_fscore_support(yt,yp,average='macro',zero_division=0)
acc=accuracy_score(yt,yp)
print('RESULTADOS VISUALES SOBRE TEST RESERVADO:')
print('Accuracy:',round(acc,3),'| Precision macro:',round(pr,3),'| Recall macro:',round(rc,3),'| F1 macro:',round(f1,3))
print('Reporte por clase:')
print(classification_report(yt,yp,target_names=CLASES,labels=list(range(len(CLASES))),zero_division=0))
matriz=confusion_matrix(yt,yp,labels=list(range(len(CLASES))))
fig,ax=plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(matriz,display_labels=CLASES).plot(ax=ax,xticks_rotation=25,colorbar=False,cmap='Blues')
ax.set_title('ConstructIA V3 — ResNet50 (conjunto de prueba)')
plt.tight_layout();plt.show()

## 7. Primera integración texto–imagen con prioridad comercial
**Se conserva la predicción SVM original** y se agrega una regla de negocio explicable: si la consulta contiene lenguaje explícito de precio o cotización, ConstructIA prioriza la intención `cotizacion`. Así documentamos por separado el resultado de ML y la intervención de reglas.

El componente visual sigue siendo ResNet50; una imagen del conjunto de prueba permite demostrar la comunicación entre componentes. **No se modifican retroactivamente las métricas SVM de 64.4% ni las métricas del experimento V3 original.** Un asesor debe confirmar máquina, tarifa y disponibilidad.


In [ ]:
import subprocess
REPO='/content/ConstructIA'
if os.path.isdir(os.path.join(REPO,'.git')):
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/chrisavm24-dot/ConstructIA.git',REPO],check=True)
if REPO+'/avance2' not in sys.path: sys.path.insert(0,REPO+'/avance2')
from nlp_modelo import entrenar_y_evaluar, predecir
from integracion_prioridades import resolver_intencion_comercial, responder_texto_imagen

modelo_texto=entrenar_y_evaluar()['pipeline']
consulta='Me gustaría conocer el precio para rentar una máquina'
registro_imagen=test_df.iloc[0]
texto_out=predecir(modelo_texto, consulta)
decision=resolver_intencion_comercial(consulta,texto_out['intencion_predicha'])
red.eval()
with torch.no_grad():
    xx=transform_eval(Image.open(registro_imagen['ruta']).convert('RGB')).unsqueeze(0).to(DEVICE)
    indice_visual=int(red(xx).argmax(1).item())
clase_visual=CLASES[indice_visual]
print('CONSULTA:',consulta)
print('OPERACIÓN 1 — SVM, predicción original:',decision['prediccion_svm'])
print('OPERACIÓN 1B — REGLA COMERCIAL:',decision['regla_aplicada'])
print('INTENCIÓN FINAL:',decision['intencion_final'])
print('OPERACIÓN 2 — IMAGEN, categoría predicha:',clase_visual)
print('OPERACIÓN 3 — RESPUESTA DE ConstructIA:')
print(responder_texto_imagen(decision,clase_visual))
display(Image.open(registro_imagen['ruta']).resize((420,300)))
print('Fuente de la imagen:',registro_imagen['pagina_fuente'])

print('\nOTRAS CONSULTAS: modelo vs. prioridad comercial')
for frase in [
    'Necesito saber el precio de una retroexcavadora',
    '¿Me pueden dar una cotización de una plataforma elevadora?',
    'No quiero operador para la máquina'
]:
    pred=predecir(modelo_texto,frase)['intencion_predicha']
    print(frase,'| SVM:',pred,'| decisión:',resolver_intencion_comercial(frase,pred)['intencion_final'])


## 8. Interpretación de V3.1 y límites
- Dataset original V3: **165 imágenes**, distribuidas en tres clases (55 por clase). V3 obtuvo **72.0% accuracy** y **F1 macro 0.698** sobre 25 imágenes. Son resultados anteriores, **no** del entrenamiento V3.1.
- El filtro V3.1 propone excluir unos pocos ejemplos vistos en las capturas. Su coincidencia se muestra en pantalla y requiere confirmar el conjunto completo mediante galerías.
- **Las métricas V3.1 solo existirán cuando se vuelva a entrenar y ejecutar esta versión**. Como ya se examinaron los resultados V3, V3.1 es una iteración exploratoria; para una medición imparcial futura se requerirán fotografías externas nuevas.
- En texto, el modelo SVM obtuvo **64.4% accuracy** sobre 45 mensajes sintéticos reservados. La regla de prioridad para cotizaciones corrige un caso de negocio en la respuesta integrada, **sin mejorar ni cambiar esa métrica del SVM**.
- La fuente Wikimedia Commons conserva autoría, licencia y página original en los manifiestos CSV. No implica validación del modelo exacto, seguridad de uso, ni disponibilidad real.
- Próximos pasos: validar todas las etiquetas, ampliar clases/datos y revisar imágenes de otros proveedores con autorización; mantener una clase «otro» y revisión humana.

**Archivos para conservar:** `origen_licencias_imagenes_curado.csv`, `exclusiones_curacion_v3_1.csv`, manifiestos de partición, capturas, informe y pesos del modelo. Se puede hacer copia con la sección siguiente.


## 9. Respaldo para conservar el trabajo en Drive (opcional)
Colab utiliza almacenamiento temporal. Esta celda **no envía datos a terceros**: solicita autorización de tu Google Drive solo si activas `GUARDAR_EN_DRIVE = True`. Se copian los pesos del modelo, los CSV de fuentes y división y el historial; las imágenes descargadas pueden volver a obtenerse usando sus URL y licencias. **No actives el respaldo antes de terminar de entrenar**.


In [ ]:
import shutil
GUARDAR_EN_DRIVE=False  # Cambia a True si deseas conservar los archivos en tu Drive.
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destino=pathlib.Path('/content/drive/MyDrive/ConstructIA - Avance 2/V3_1')
    destino.mkdir(parents=True,exist_ok=True)
    for archivo in [
        'constructia_resnet50.pt','origen_licencias_imagenes.csv',
        'origen_licencias_imagenes_curado.csv',
        'exclusiones_curacion_v3_1.csv',
        'entrenamiento_manifest.csv','validación_manifest.csv','prueba_manifest.csv'
    ]:
        src=ROOT/archivo
        if src.exists():
            shutil.copy2(src,destino/archivo)
            print('Respaldo:',archivo)
        else:
            print('No existe todavía:',archivo)
    print('Respaldo realizado en',destino)
else:
    print('Respaldo desactivado. Los archivos de /content pueden perderse al finalizar la sesión.')
